# core

> Alignment state context, getters, and updaters for route handlers

In [ ]:
#| default_exp routes.core

In [ ]:
#| export
from typing import Any, Dict, List, Optional, NamedTuple

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_transcript_vad_align.models import (
    VADChunk, AlignmentStepState
)

# Type alias for state store (duck-typed, accepts any implementation with get_state/update_state)
WorkflowStateStore = SQLiteWorkflowStateStore

# Debug flag for alignment state tracing (set False in production)
DEBUG_ALIGN_STATE = False

## AlignContext

Common state values loaded by alignment handlers in a single call.

In [ ]:
#| export
class AlignContext(NamedTuple):
    """Common alignment state values loaded by handlers."""
    chunk_dicts: List[Dict[str, Any]]  # Serialized VAD chunks
    focused_chunk_index: int  # Currently focused VAD chunk index
    visible_count: int  # Number of visible cards in viewport
    is_auto_mode: bool  # Whether card count is in auto-adjust mode
    card_width: int  # Card stack width in rem
    media_path: Optional[str]  # Path to first audio file (backward compat)
    media_paths: List[str]  # Ordered list of all audio file paths
    audio_duration: Optional[float]  # Total audio duration
    auto_navigate: bool  # Auto-advance to next chunk when audio finishes
    playback_speed: float  # Pitch-preserving playback speed (1.0 = normal)

## State Management

Functions for reading and writing alignment state via the workflow state store.

In [ ]:
#| export
def _get_alignment_state(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    session_id:str,  # Session identifier
) -> AlignmentStepState:  # Typed alignment step state
    """Get the alignment step state from the workflow state store."""
    state = state_store.get_state(workflow_id, session_id)
    step_states = state.get("step_states", {})
    return step_states.get("alignment", {})

def _get_selection_state(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    session_id:str,  # Session identifier
) -> Dict[str, Any]:  # Selection step state dictionary
    """Get the selection step state (Phase 1) from the workflow state store."""
    state = state_store.get_state(workflow_id, session_id)
    step_states = state.get("step_states", {})
    return step_states.get("selection", {})

def _load_alignment_context(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    session_id:str,  # Session identifier
) -> AlignContext:  # Loaded context with all common alignment state
    """Load commonly-needed alignment state values in a single call."""
    state = _get_alignment_state(state_store, workflow_id, session_id)
    return AlignContext(
        chunk_dicts=state.get("vad_chunks", []),
        focused_chunk_index=state.get("focused_chunk_index", 0),
        visible_count=state.get("visible_count", 5),
        is_auto_mode=state.get("is_auto_mode", False),
        card_width=state.get("card_width", 40),
        media_path=state.get("media_path"),
        media_paths=state.get("media_paths", []),
        audio_duration=state.get("audio_duration"),
        auto_navigate=state.get("auto_navigate", False),
        playback_speed=state.get("playback_speed", 1.0),
    )

def _update_alignment_state(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    session_id:str,  # Session identifier
    vad_chunks=None,  # Updated VAD chunks (serialized)
    focused_chunk_index=None,  # Updated focused chunk index
    is_initialized=None,  # Initialization flag
    visible_count=None,  # Visible card count
    is_auto_mode=None,  # Auto-adjust mode flag
    card_width=None,  # Card stack width in rem
    media_path=None,  # First audio file path (backward compat)
    media_paths=None,  # Ordered list of all audio file paths
    audio_duration=None,  # Audio duration
    auto_navigate=None,  # Auto-navigate flag
    playback_speed=None,  # Pitch-preserving playback speed
) -> None:
    """Update the alignment step state in the workflow state store."""
    if DEBUG_ALIGN_STATE:
        print(f"[ALIGN_STATE] _update_alignment_state called")

    # Read full state to preserve sibling step states
    workflow_state = state_store.get_state(workflow_id, session_id)

    step_states = workflow_state.get("step_states", {})
    align_state = step_states.get("alignment", {})

    # Update only provided fields
    if vad_chunks is not None:
        align_state["vad_chunks"] = vad_chunks
    if focused_chunk_index is not None:
        align_state["focused_chunk_index"] = focused_chunk_index
    if is_initialized is not None:
        align_state["is_initialized"] = is_initialized
    if visible_count is not None:
        align_state["visible_count"] = visible_count
    if is_auto_mode is not None:
        align_state["is_auto_mode"] = is_auto_mode
    if card_width is not None:
        align_state["card_width"] = card_width
    if media_path is not None:
        align_state["media_path"] = media_path
    if media_paths is not None:
        align_state["media_paths"] = media_paths
    if audio_duration is not None:
        align_state["audio_duration"] = audio_duration
    if auto_navigate is not None:
        align_state["auto_navigate"] = auto_navigate
    if playback_speed is not None:
        align_state["playback_speed"] = playback_speed

    step_states["alignment"] = align_state
    workflow_state["step_states"] = step_states

    state_store.update_state(workflow_id, session_id, workflow_state)

## Conversion Helpers

In [ ]:
#| export
def _to_vad_chunks(
    chunk_dicts:List[Dict[str, Any]]  # Serialized VAD chunk dictionaries
) -> List[VADChunk]:  # List of VADChunk objects
    """Convert chunk dictionaries to VADChunk objects."""
    return [VADChunk.from_dict(c) for c in chunk_dicts]

def _build_card_stack_state(
    ctx:AlignContext,  # Loaded alignment context
    active_mode:str=None,  # Current interaction mode name (unused for alignment)
) -> CardStackState:  # Card stack state for library functions
    """Build a CardStackState from alignment context for library functions."""
    return CardStackState(
        focused_index=ctx.focused_chunk_index,
        visible_count=ctx.visible_count,
        card_width=ctx.card_width,
        active_mode=active_mode,
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()